# Train/val/test split, overfitting, regularization

*0.2 Math / ML basics · run **Setup** first*

## Setup

Settings, a configured client, an `embed()` helper and numpy. Every cell below uses them.

In [1]:
"""Shared setup: typed settings, a configured client, numpy, and a print helper."""

import json

import numpy as np
from dotenv import find_dotenv
from openai import OpenAI
from pydantic import Field, SecretStr
from pydantic_settings import BaseSettings, SettingsConfigDict

np.set_printoptions(precision=4, suppress=True)


class Settings(BaseSettings):
    model_config = SettingsConfigDict(env_file=find_dotenv(), extra="ignore")

    openai_api_key: SecretStr
    openai_model: str = "gpt-4o-mini"
    embedding_model: str = "text-embedding-3-small"
    request_timeout_seconds: float = Field(default=30, gt=0)
    max_retries: int = Field(default=2, ge=0, le=5)


settings = Settings()
client = OpenAI(
    api_key=settings.openai_api_key.get_secret_value(),
    timeout=settings.request_timeout_seconds,
    max_retries=settings.max_retries,
)


def embed(texts: list[str]) -> np.ndarray:
    """Embed texts with the configured model; returns one row per text."""
    response = client.embeddings.create(model=settings.embedding_model, input=texts)
    rows = []
    for item in response.data:
        rows.append(item.embedding)
    return np.array(rows)


def show(title: str, value) -> None:
    print(title)
    print(json.dumps(value, indent=2, ensure_ascii=False, default=str))


print("chat model:", settings.openai_model, "| embedding model:", settings.embedding_model)

chat model: gpt-4o-mini | embedding model: text-embedding-3-small


### Train/val/test split

> **Problem.** A model scores 98% on the data it was trained on and 71% on the first week of real traffic. The 98% measured memory, not skill — the evaluation used rows the model had already seen.

**Idea.** Hold data back: train on one part, choose settings on a second, report on a third the model never touched.

**Use when** any model you will make a decision about — including prompt and RAG evaluations.  
**Not when** —.

```
all data  ──▶  train 60%  ──▶ fit the model
          ──▶  val   20%  ──▶ pick settings (degree, alpha, prompt version)
          ──▶  test  20%  ──▶ report once, never tune on it
```

**How it works.**
1. `train_test_split(X, y, test_size=0.4, random_state=0)` splits off 40% with a fixed seed so the split is reproducible.
2. A second split divides that 40% into validation and test.
3. The model is fit on train; the three error numbers show train lowest, validation and test slightly higher — the honest picture.
4. The leaky model, fit on everything, reports a flattering test error because those rows were in training.

| | what happens | result |
|:--|:--|:--|
| ✗ leaky | fit on all rows, score on test | flattering number |
| ✓ split | fit on train, score on untouched test | the number you will see in production |

**Production code and its real output**

In [2]:
# Train / validation / test split — sklearn does it, with a fixed seed so it is reproducible.
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(0)
X = rng.uniform(-3, 3, size=(300, 1))
y = np.sin(X[:, 0]) + rng.normal(scale=0.3, size=300)

X_train, X_rest, y_train, y_rest = train_test_split(X, y, test_size=0.4, random_state=0)
X_val, X_test, y_val, y_test = train_test_split(X_rest, y_rest, test_size=0.5, random_state=0)
print("train", len(X_train), "| validation", len(X_val), "| test", len(X_test))

model = LinearRegression().fit(X_train, y_train)
for name, X_part, y_part in [
    ("train", X_train, y_train),
    ("validation", X_val, y_val),
    ("test", X_test, y_test),
]:
    print(f"{name:<11} mse {np.mean((model.predict(X_part) - y_part) ** 2):.3f}")

# Leakage: fitting on everything then scoring on 'test' rows that were in training flatters the number.
leaky = LinearRegression().fit(X, y)
print(
    
        f"leaky       mse {np.mean((leaky.predict(X_test) - y_test) ** 2):.3f}  (test rows were in "
        f"training)"
    
)
assert len(X_train) == 180 and len(X_test) == 60

train 180 | validation 60 | test 60
train       mse 0.225
validation  mse 0.286
test        mse 0.277
leaky       mse 0.276  (test rows were in training)


**What the output shows.** 180 / 60 / 60 rows; validation and test errors sit above training error, and the leaky model's test score is visibly better than it deserves.

**In practice**
- **split by time or group** — random splits leak when rows are correlated (same user, same day); split by the unit that matters.
- **stratify** — `stratify=y` keeps class ratios equal across splits for classification.
- **test once** — every time you look at the test score and change something, it becomes a validation set; keep a final holdout.
- **same for prompts** — a prompt tuned on the examples you eyeball is overfit; keep an unseen eval set for prompts and RAG too (layer 12).

**Alternatives** — k-fold cross-validation for small data · time-based splits for anything with a date

**Terms** — *holdout*: data kept out of training · *leakage*: test information reaching the model · *random_state*: the seed that makes a split repeatable


### overfitting

> **Problem.** Each added feature, layer or epoch makes the training score better, so the natural instinct is to keep adding. At some point the model starts memorising noise, and the score on new data gets worse while the training score keeps improving.

**Idea.** Capacity beyond what the data supports fits the noise; validation error is the signal that says when to stop.

**Use when** choosing model size, epochs, polynomial degree, tree depth — any capacity knob.  
**Not when** —.

```
capacity →      1        3        5        9       15
train mse     0.26     0.06     0.06     0.05     0.04   ↓ always
val   mse     0.33     0.18     0.25     1.00     2.50   ↓ then ↑   ← stop at the minimum
```

**How it works.**
1. Polynomial features of degree 1 to 15 give the same linear model increasing capacity.
2. Training error falls at every degree — more capacity can always fit the training points better.
3. Validation error falls to degree 3 then rises sharply; from there the extra capacity is fitting noise.
4. The best degree is chosen by the validation minimum, never by the training score.

| | what happens | result |
|:--|:--|:--|
| ✗ degree 15 | train mse tiny | val mse large — memorised noise |
| ✓ degree 3 | train mse small | val mse smallest — generalises |

**Production code and its real output**

In [3]:
# Overfitting — more capacity lowers training error forever; validation error turns back up.
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

rng = np.random.default_rng(0)
X = rng.uniform(-3, 3, size=(60, 1))
y = np.sin(X[:, 0]) + rng.normal(scale=0.3, size=60)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.5, random_state=0)

print(f"{'degree':<8}{'train mse':>10}{'val mse':>10}")
results = {}
for degree in [1, 3, 5, 9, 15]:
    model = make_pipeline(PolynomialFeatures(degree), StandardScaler(), LinearRegression()).fit(
        X_train, y_train
    )
    results[degree] = (
        np.mean((model.predict(X_train) - y_train) ** 2),
        np.mean((model.predict(X_val) - y_val) ** 2),
    )
    print(f"{degree:<8}{results[degree][0]:>10.3f}{results[degree][1]:>10.3f}")
best = min(results, key=lambda d: results[d][1])
print("best degree by validation:", best)
assert results[15][0] < results[1][0] and results[15][1] > results[best][1]

degree   train mse   val mse
1            0.263     0.329
3            0.063     0.175
5            0.060     0.253
9            0.047     1.001
15           0.031   512.054
best degree by validation: 3


**What the output shows.** Training error decreased monotonically with degree while validation error bottomed at degree 3 and exploded by degree 15.

**In practice**
- **early stopping** — for neural networks the capacity knob is epochs; stop when validation loss stops improving.
- **more data beats less capacity** — if you can get data, that moves the whole curve; regularization only bends it.
- **watch the gap** — a large train/validation gap is the symptom; log both every epoch.
- **LLM fine-tuning** — 3 epochs on 1,000 examples overfits fast — validation loss on held-out examples, always (layer 1.3).

**Alternatives** — regularization (next item) · dropout and weight decay for neural nets · simpler models

**Terms** — *capacity*: how complex a function the model can represent · *generalise*: perform well on unseen data · *epoch*: one pass over the training data


### regularization

> **Problem.** The high-capacity model is the right choice — it captures the real curve — but left alone it also fits the noise. You want its flexibility without the memorisation.

**Idea.** Add a penalty on large weights to the loss, so the model uses its capacity only where the data justifies it.

**Use when** any model whose validation error is worse than its training error by a meaningful margin.  
**Not when** the model is underfitting — the penalty makes that worse.

```
loss = error on data  +  α · Σ w²
                             ▲
        α = 0    fits noise, weights huge
        α = 1    smooth fit, weights small     ← chosen by validation
        α = 100  too stiff, underfits
```

**How it works.**
1. `Ridge(alpha=α)` minimises the data error plus α times the sum of squared weights.
2. The same degree-15 features are used; without a penalty the validation error is large and the weights are huge.
3. As α grows the weights shrink, the fit smooths, and validation error falls — until α is so large the model can no longer follow the curve.
4. The best α is picked by validation error, exactly like the best degree.

| | what happens | result |
|:--|:--|:--|
| ✗ α=0 | degree 15, free weights | large val error, huge weights |
| ✓ α≈1 | same features, penalised | smallest val error, small weights |
| ✗ α=100 | over-penalised | underfits |

**Production code and its real output**

In [4]:
# Regularization — Ridge adds α·Σw² to the loss, pulling a high-capacity model toward simpler fits.
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

rng = np.random.default_rng(0)
X = rng.uniform(-3, 3, size=(60, 1))
y = np.sin(X[:, 0]) + rng.normal(scale=0.3, size=60)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.5, random_state=0)
degree = 15

plain = make_pipeline(PolynomialFeatures(degree), StandardScaler(), LinearRegression()).fit(
    X_train, y_train
)
plain_val = np.mean((plain.predict(X_val) - y_val) ** 2)
print(f"degree 15, no penalty   val mse {plain_val:.3f}")

print(f"{'alpha':<8}{'val mse':>9}{'max |w|':>9}")
results = {}
for alpha in [0.001, 0.1, 1.0, 10.0]:
    model = make_pipeline(PolynomialFeatures(degree), StandardScaler(), Ridge(alpha=alpha)).fit(
        X_train, y_train
    )
    weights = model.named_steps["ridge"].coef_
    results[alpha] = np.mean((model.predict(X_val) - y_val) ** 2)
    print(f"{alpha:<8}{results[alpha]:>9.3f}{np.abs(weights).max():>9.3f}")
assert min(results.values()) < plain_val

degree 15, no penalty   val mse 512.054
alpha     val mse  max |w|
0.001       4.630    5.422
0.1         0.244    1.475
1.0         0.604    1.089
10.0        0.406    0.477


**What the output shows.** The unpenalised degree-15 model had the worst validation error; a moderate α cut it sharply and shrank the largest weight by orders of magnitude.

**In practice**
- **weight decay** — the neural-network name for the same L2 penalty; AdamW applies it correctly — use it by default.
- **dropout** — randomly zeroing activations during training is the other standard regulariser for deep nets.
- **scale features first** — the penalty treats all weights equally, so features must be on the same scale (`StandardScaler`).
- **L1 for selection** — Lasso (L1) drives weights to exactly zero — useful when you want fewer features.

**Alternatives** — dropout · early stopping · data augmentation · smaller model

**Terms** — *α (alpha)*: how strong the penalty is · *L2 / ridge*: penalty on squared weights · *weight decay*: the same idea in neural-network optimisers
